In [1]:
import numpy as np
import matplotlib.pyplot as plt

1k Начнём со второго пункта. Рассмотрим многочлен Уилкинсона

P (x) = (x − 1)(x − 2)...(x − 20) = x20 − 210x19 + 20615x18 + . . .

Очевидно, корнями его являются x1 = 1, x2 = 2, . . ., x20 = 20. Выполните
в Matlab команду p=poly(1:20), которая позволяет получить коэффи-
циенты полинома, корнями которого является аргумент функции. Набе-
рите roots(p) и убедитесь, что корни полинома найдены верно. Измени-
те значение, например, второго коэффициента на малую величину 10−7
(составляет ≈ 5 · 10−8 % от числа) и снова выполните эту команду - по-
ловина корней стала комплексными числами! Исходная задача оказалась
неустойчивой к входным данным (их малое изменение ведет к сильному
изменению решения), в результате чего получился абсурдный результат.

In [21]:
roots = np.arange(1, 21 , dtype=float)
coefs = np.poly(roots)
roots

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20.])

In [22]:
coefs

array([ 1.00000000e+00, -2.10000000e+02,  2.06150000e+04, -1.25685000e+06,
        5.33279460e+07, -1.67228082e+09,  4.01717716e+10, -7.56111184e+11,
        1.13102770e+13, -1.35585183e+14,  1.30753501e+15, -1.01422999e+16,
        6.30308121e+16, -3.11333643e+17,  1.20664780e+18, -3.59997952e+18,
        8.03781182e+18, -1.28709312e+19,  1.38037598e+19, -8.75294804e+18,
        2.43290201e+18])

In [15]:
print("Степень:", len(coefs) - 1)
print("Второй коэффициент (при x^19):", coefs[1])

Степень: 20
Второй коэффициент (при x^19): -210.0


In [25]:
coef_pert = coefs.copy()
coef_pert[1] += 1e-6
coef_pert

array([ 1.00000000e+00, -2.09999999e+02,  2.06150000e+04, -1.25685000e+06,
        5.33279460e+07, -1.67228082e+09,  4.01717716e+10, -7.56111184e+11,
        1.13102770e+13, -1.35585183e+14,  1.30753501e+15, -1.01422999e+16,
        6.30308121e+16, -3.11333643e+17,  1.20664780e+18, -3.59997952e+18,
        8.03781182e+18, -1.28709312e+19,  1.38037598e+19, -8.75294804e+18,
        2.43290201e+18])

In [26]:
coefs[1], coef_pert[1]

(np.float64(-210.0), np.float64(-209.999999))

In [27]:
roots_pert = np.roots(coef_pert)
roots_pert

array([21.30248897+1.56716845j, 21.30248897-1.56716845j,
       18.5027532 +3.6003971j , 18.5027532 -3.6003971j ,
       15.16511631+3.76123653j, 15.16511631-3.76123653j,
       12.48656743+2.88273411j, 12.48656743-2.88273411j,
       10.52262054+1.71953973j, 10.52262054-1.71953973j,
        9.04495188+0.59505952j,  9.04495188-0.59505952j,
        7.94850424+0.j        ,  7.00255525+0.j        ,
        5.99994248+0.j        ,  5.00000034+0.j        ,
        4.00000004+0.j        ,  3.        +0.j        ,
        2.        +0.j        ,  1.        +0.j        ])

In [28]:
complex_cnt = np.sum(np.abs(roots_pert.imag) > 1e-6)
complex_cnt

np.int64(12)

k Знание этих фактов позволяет протестировать компьютер на по-
грешность вычислении. Найдем значение выражения (1+ε−1)/ε
при ε = 2^(−n) ,
n = 0, 1, 2, . . .. С точки зрения аналитической математики значение этого

In [36]:
for n in range(50,60):
    eps = np.ldexp(np.float64(1.0), -n)  
    print(n, ((np.float64(1.0) + eps) - np.float64(1.0)) / eps)

50 1.0
51 1.0
52 1.0
53 0.0
54 0.0
55 0.0
56 0.0
57 0.0
58 0.0
59 0.0


2k Рассмотрим погрешности численных методов. Построим алгоритм
вычисления интеграла In =
R1
n x−1
dx, n
0x e
= 1, 2, 3, . . . Интегрируя по
частям, находим:
R1
R1
I1 = xex−1 dx = xex−1 |10 − ex−1 dx = 1e
0
0
R1
R1
I2 = x2 ex−1 dx = x2 ex−1 |10 − 2 xex−1 dx = 1 − 2I1
0
···
R1
0
n x−1
In = x e
0
dx = xn ex−1 |10 − n
R1
0
xn−1 ex−1 dx = 1 − nIn−1
Вычислите значения интегралов, до n = 30. Заметьте, что подинтеграль-
ная функция на всем отрезке интегрирования неотрицательна, следова-
тельно, и значение интеграла — положительное число. Более того, подын-
тегральная функция на данном интервале ограничена функцией y = 1,
т.е. значение интеграла не может превышать единицы. Посмотрите, как
это согласуется с полученными результатами. Попытайтесь на основе по-
лученных сведений о погрешностях объяснить данный феномен.

In [37]:
import numpy as np
from scipy.integrate import quad

In [38]:
def I_quad(n):
    f = lambda x: (x**n) * np.exp(x-1)
    val, err = quad(f, 0.0, 1.0)
    return val


In [39]:
N = 30
I_ref = np.array([I_quad(n) for n in range(N+1)], dtype=np.float64)
print(I_ref)

[0.63212056 0.36787944 0.26424112 0.20727665 0.17089341 0.14553294
 0.12680236 0.1123835  0.10093197 0.09161229 0.08387707 0.07735223
 0.07177325 0.0669477  0.06273216 0.05901754 0.05571935 0.05277112
 0.05011985 0.04772276 0.04554488 0.04355743 0.04173644 0.04006181
 0.03851655 0.03708621 0.03575842 0.03452253 0.03336929 0.03229068
 0.03127967]


Напишите функцию вычисления значения синуса в виде конечной
суммы ряда2 (sin x =
P∞
2k+1
k x
k=0 (−1) (2k+1)! ). По признаку Лейбница погреш-

In [41]:
import math

In [49]:
sum = np.float128(0.0)
x = np.float128(0.001)

for k in range(0, 100):
    sum += (-1)**(k) * x**(2*k+1) / math.factorial(2*k+1)

print(np.sin(x), sum)

0.0009999998333333416875 0.0009999998333333416874
